In [1]:
# Logistic Regression Implementation
import pandas as pd
import sklearn
import numpy as np
import scipy
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline

In [2]:
df = pd.read_csv("results/assignment3/affinity/sorted_genes.csv")
df

,Gene,SRR2534402,SRR2534403,SRR2534404,SRR2534405,SRR2534406,SRR2534407,SRR2534408,SRR2534409,SRR2534410,...,SRR2535038,SRR2535039,SRR2535040,SRR2535041,SRR2535042,SRR2535043,SRR2535044,SRR2535045,SRR2535046,SRR2535047
0,Lars2,18.686778,17.942373,19.279816,19.727014,20.043443,18.466178,19.021748,18.138422,90.305486,...,11.997342,11.445452,9.899983,8.863214,9.204295,9.155986,10.286630,8.943861,9.934873,9.645423
1,Mir6236,40.008331,40.724974,41.262309,41.580819,40.355943,40.469911,40.285823,41.452146,80.438028,...,35.227715,34.002301,41.843580,42.504179,41.085445,42.238744,41.326141,41.961827,41.514807,41.713337
2,Mup15,85.497160,84.877921,84.382044,85.497160,85.497160,84.877921,84.877921,85.497160,76.617575,...,80.001721,79.282446,98.170073,98.170073,97.362222,98.170073,97.362222,97.362222,98.170073,99.066884
3,Mup17,74.490667,75.350819,75.050092,75.050092,74.749365,75.050092,75.350819,74.749365,71.182024,...,77.984974,76.954759,93.417512,94.029779,93.417512,94.029779,93.417512,94.029779,94.029779,93.417512
4,Grip2,57.645500,47.460108,29.807702,28.382645,39.775930,29.680578,33.569053,24.718278,52.866789,...,59.079150,67.010969,58.625124,55.672795,39.645982,43.867946,46.822504,42.238744,45.647330,53.196817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37766,Krtap2-21,0.236804,0.236517,0.237815,0.237395,0.234727,0.236399,0.236287,0.237697,0.231434,...,0.219176,0.219518,0.224586,0.221831,0.225496,0.222731,0.222293,0.225023,0.221662,0.225210
37767,Or1o2,0.236804,0.236517,0.237815,0.237395,0.234727,0.236399,0.236287,0.237697,0.231434,...,0.219176,0.219518,0.224586,0.221831,0.225496,0.222731,0.222293,0.225023,0.221662,0.225210
37768,Gm15290,0.236804,0.236517,0.237815,0.237395,0.234727,0.236399,0.236287,0.237697,0.231434,...,0.219176,0.219518,0.224586,0.221831,0.225496,0.222731,0.222293,0.225023,0.221662,0.225210
37769,Vmn2r51,0.236804,0.236517,0.237815,0.237395,0.234727,0.236399,0.236287,0.237697,0.231434,...,0.219176,0.219518,0.224586,0.221831,0.225496,0.222731,0.222293,0.225023,0.221662,0.225210


In [3]:
# Make a list for the experiment labels
hierarchical_results = pd.read_csv('results/assignment3/hierarchical/cluster_labels.csv')
sample_groups = []
for sample in hierarchical_results["sample_id"]:
    sample_groups.append(int(sample[3:]) <= 2534782) # if this conditional is true, the sample is in the AL group, else RF group

In [4]:
hierarchical_cluster_groups = hierarchical_results["hierarchical_g5000_k4"]
hierarchical_cluster_groups

0      1
1      1
2      1
3      1
4      1
      ..
541    3
542    3
543    3
544    3
545    3
Name: hierarchical_g5000_k4, Length: 546, dtype: int64

In [5]:
X = df.drop(columns='Gene')[:5000].T.to_numpy() # Becomes a 2d array for input
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000)) # choose 10000 iterations to ensure convergence

# getting labels and probabilities from logistic regression
cv = StratifiedKFold(5, shuffle=True, random_state=42)
labels = cross_val_predict(clf, X, sample_groups, cv=cv)
probs  = cross_val_predict(clf, X, sample_groups, cv=cv, method="predict_proba")
roc_auc_score(sample_groups, probs[:, 1])

1.0

In [6]:
labels_cluster = cross_val_predict(clf, X, hierarchical_cluster_groups, cv=cv)
probs_cluster = cross_val_predict(clf, X, hierarchical_cluster_groups, cv=cv, method="predict_proba")
roc_auc_score(hierarchical_cluster_groups, probs_cluster, multi_class="ovr")

1.0

In [7]:
# how gene count affects ROC
gene_counts = [10, 100, 1000, 10000]
print("Experimental Labels:")
for gc in gene_counts:
    X = df.drop(columns='Gene')[:gc].T.to_numpy()
    probs  = cross_val_predict(clf, X, sample_groups, cv=cv, method="predict_proba")
    print("Gene Count:", gc, "| AUC:", roc_auc_score(sample_groups, probs[:, 1]))

print("Cluster Labels")
for gc in gene_counts:
    X = df.drop(columns='Gene')[:gc].T.to_numpy()
    probs_cluster = cross_val_predict(clf, X, hierarchical_cluster_groups, cv=cv, method="predict_proba")
    print("Gene Count:", gc, "| AUC:", roc_auc_score(hierarchical_cluster_groups, probs_cluster, multi_class="ovr"))

Experimental Labels:
Gene Count: 10 | AUC: 0.9793433090283592
Gene Count: 100 | AUC: 1.0
Gene Count: 1000 | AUC: 1.0
Gene Count: 10000 | AUC: 1.0
Cluster Labels
Gene Count: 10 | AUC: 0.9377611380442941
Gene Count: 100 | AUC: 1.0
Gene Count: 1000 | AUC: 1.0
Gene Count: 10000 | AUC: 1.0


In [8]:
X = df.drop(columns='Gene')[:5000].T.to_numpy()
clf.fit(X, sample_groups)
importance = np.argsort(clf[-1].coef_[0])[::-1][:10]
# Top 10 important genes:
df["Gene"][importance]

2434           Picalm
469             Abcc6
4221             Cib1
2467             Mrc1
1070              Ttn
3322            Actr2
872             Mup13
950     0610005C13Rik
1110             Ctsh
1076            Pcsk6
Name: Gene, dtype: str

In [12]:
pd.DataFrame([labels, labels_cluster, hierarchical_cluster_groups]).to_csv("results/assignment4/logistic_results.csv", index=False)